In [ ]:
# importing the required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler , LabelEncoder ,StandardScaler ,OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge , Lasso ,ElasticNet ,LinearRegression , LogisticRegression 
from sklearn.ensemble import RandomForestRegressor , GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor 
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score ,mean_squared_error

# Set random seed for reproducibility
np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 5)

In [ ]:
# importing the dataset from kaggle path /kaggle/input/competitions/midad-ml-competition/train_set_.csv
data = pd.read_csv("/kaggle/input/competitions/midad-ml-competition/train_set_.csv",sep=';')

# checking the first 5 rows of the dataset
data.head()

# checking the shape of the dataset
data.shape

# checking the info of the dataset
data.info()

# checking the statistical summary of the dataset
data.describe()

In [ ]:
# drop ID column
data=data.drop(columns=["ID"], axis=1)
data.head()

### preprocessing the dataset

In [ ]:
# checking for missing values
data.isnull().sum()

In [ ]:
# checking for duplicate values
data.duplicated().sum()

In [ ]:
# present the data with filter feuling type = 'Electric'
data[data['Fuel_Type'] == 'Electric']

we see the 2 missing values in Mileage come with Electric Fuel Tupe

In [ ]:
kmkgCNG = 0
kmkgpetrol = 0
kmkgDiesel = 0
kmkgLPG = 0
kmplCNG = 0
kmplPetrol = 0
kmplDiesel = 0
kmplLPG = 0

for i , j in zip(data.Mileage , data.Fuel_Type):
    if str(i).endswith("km/kg") and j == "CNG":
        kmkgCNG+=1
    elif str(i).endswith("km/kg") and j == "Petrol":
        kmkgpetrol+=1
    elif str(i).endswith("km/kg") and j == "Diesel":
        kmkgDiesel+=1
    elif str(i).endswith("km/kg") and j == "LPG":
        kmkgLPG+=1
    elif str(i).endswith("kmpl") and j == "CNG":
        kmplCNG+=1
    elif str(i).endswith("kmpl") and j == "Petrol":
        kmplPetrol+=1
    elif str(i).endswith("kmpl") and j == "Diesel":
        kmplDiesel+=1
    elif str(i).endswith("kmpl") and j == "LPG":
        kmplLPG+=1
print('The number of rows with Km/Kg and CNG : {} '.format(kmkgCNG))
print('The number of rows with Km/Kg and Petrol : {} '.format(kmkgpetrol))
print('The number of rows with Km/Kg and Diesel : {} '.format(kmkgDiesel))
print('The number of rows with Km/Kg and LPG : {} '.format(kmkgLPG))
print('The number of rows with kmpl and CNG : {} '.format(kmplCNG))
print('The number of rows with Km/L and Petrol : {} '.format(kmplPetrol))
print('The number of rows with Km/L and Diesel : {} '.format(kmplDiesel))
print('The number of rows with Km/L and LPG : {} '.format(kmplLPG))
print('The total number of rows : {} '.format(kmkgCNG + kmkgpetrol + kmkgDiesel + kmkgLPG + kmplCNG + kmplPetrol + kmplDiesel + kmplLPG))


### we have Km/Kg unit with CNG and LPG fuel type and it's so fewer than kmpl
so we should convert Km/Kg to Km/L and after that we use Mileage's column to float

1 kg of CNG occupies roughly 1.4 to 1.5 Liters of volume equivalent under normal dispenser conditions.

so our formula will be:

###     1 KmpL = 1 KmpKg / 1.4 (CNG)

1 Kg of LPG occupies approximately 1.84 Liters of volume under standard automotive fuel tank pressures.

so our formula will be:

###     1 KmpL = 1 KmpKg / 1.84 (LPG)

In [ ]:
# 1. Clean the string text and extract raw numerical values
def convert_mileage(row):
    mileage_str = str(row["Mileage"]).strip().lower()

    # Handle missing, null, or invalid entries safely
    if (
        mileage_str == "nan"
        or mileage_str == ""
        or mileage_str == "null"
    ):
        return np.nan

    # Extract numerical part from strings like "26.6 km/kg" or "19.67 kmpl"
    try:
        numeric_value = float(mileage_str.split()[0])
    except (ValueError, IndexError):
        return np.nan

    # 2. Apply fuel-specific logic
    fuel_type = str(row["Fuel_Type"]).strip().upper()

    if fuel_type == "CNG":
        # Convert km/kg to kmpl baseline by dividing by 1.4
        return round(numeric_value / 1.4, 2)
    
    elif fuel_type == "LPG":
        # Convert km/kg to kmpl baseline by dividing by 1.84
        return round(numeric_value / 1.84, 2)

    elif fuel_type in ["PETROL", "DIESEL"]:
        # Petrol and Diesel are already natively in kmpl
        return round(numeric_value, 2)

    return numeric_value

In [ ]:
data["Mileage"] = data.apply(convert_mileage, axis=1)
data

we remember there is no Mileage value with electric fuel type
so we should fill it with 0

In [ ]:
# filling missing values for Mileage with Electric fuel tupe by 0
data.loc[(data['Fuel_Type'] == 'Electric') & (data['Mileage'].isna()), 'Mileage'] = 0

In [ ]:
data.isnull().sum()

In [ ]:
# checking for missing values in Engine
data[data['Engine'].isnull()]

back to previous 2 cells there are 35 values missing in Engine and power
but if we see the price it look like the cars didn't have an engine (like missing or broken) so that's why the price is low

for the first verion of the model we will egnore these rows

In [ ]:
# drop missing engine rows
data = data.dropna(subset=['Engine'])
data.isnull().sum()

now there is no missing values in engine and power but wait....

In [ ]:
import numpy as np
import pandas as pd

# 1. Clean Engine column: Remove ' CC' and convert to float
data["Engine"] = (
    data["Engine"]
    .str.replace(" CC", "", case=False)
    .str.strip()
)
data["Engine"] = pd.to_numeric(data["Engine"], errors="coerce")

# 2. Clean Power column: Remove ' bhp', fix 'null bhp' anomalies, convert to float
data["Power"] = (
    data["Power"]
    .str.replace(" bhp", "", case=False)
    .str.strip()
)
# The Kaggle dataset specifically has strings that say 'null' inside the text
data["Power"] = data["Power"].replace("null", np.nan)
data["Power"] = pd.to_numeric(data["Power"], errors="coerce")
data


In [ ]:
data.isnull().sum()

there is also 106 values from power column hiding under name of "null bhp"
and that is a big number to drop se we need to fill it

we will fill the missing values depending on barnd , model and engine

In [ ]:
# Extract Brand (1st word) and Model (2nd word) from the Name column
data["Brand"] = data["Name"].apply(lambda x: str(x).split()[0])
data["Model"] = data["Name"].apply(lambda x: " ".join(str(x).split()[1:]))

In [ ]:
# Impute missing Power using the median power of cars with the same Model and Engine size
data["Power"] = data.groupby(["Model", "Engine"])["Power"].transform(lambda x: x.fillna(x.median()))

# Secondary fallback: If some models are unique, group by Brand and Engine size instead
data["Power"] = data.groupby(["Brand", "Engine"])["Power"].transform(lambda x: x.fillna(x.median()))

# Final global safety net fallback: Check if any nulls remain and fill with general median
data["Power"] = data["Power"].fillna(data["Power"].median())
data


In [ ]:
data.isnull().sum()

great now we only have 6 values missing from seats column

### Drop it!!!

In [ ]:
# drop missing values from Seats
data = data.dropna(subset=['Seats'])
data.isnull().sum()

# drop column name becouse we replaced by model and brand
data=data.drop(columns=["Name"])
data

Categorical columns were transformed using One-Hot Encoding to make them usable by machine learning algorithms.

In [ ]:
# for col in ['Transmission', 'Fuel_Type', 'Location', 'Owner_Type', 'Brand', 'Model']:
#     le_trans = LabelEncoder()
#     data[col] = le_trans.fit_transform(data[col].astype(str))
# data

In [ ]:
cat_cols = ['Transmission', 'Fuel_Type', 'Location', 'Owner_Type', 'Brand', 'Model']
num_cols = ['Year', 'Kilometers_Driven', 'Mileage', 'Engine', 'Power', 'Seats','Price']

# 1. Initialize OHE directly
ohe = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

# 2. Transform ONLY the categorical columns
encoded_cats = ohe.fit_transform(data[cat_cols])

# 3. Turn the encoded matrix back into a DataFrame with column names
encoded_df = pd.DataFrame(encoded_cats, columns=ohe.get_feature_names_out(cat_cols), index=data.index)

# 4. Manually stitch it together with your continuous numeric columns
data = pd.concat([encoded_df, data[num_cols]], axis=1)
data

# Training the model

The dataset was divided into training and testing sets, with 80% used for training and 20% used for evaluating the model's performance.

In [ ]:
X =data.drop('Price', axis=1)   
y = data['Price']
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
)

Feature scaling was applied using MinMaxScaler to normalize the data within a fixed range, ensuring all features contribute equally to the model.

In [ ]:
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

shape of train and test

In [32]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(4645, 1841)
(1162, 1841)
(4645,)
(1162,)


ridge

In [ ]:
ridge_pipeline = Pipeline(
    steps=[
        ('ridge', Ridge(alpha=1.0))
    ]
)
ridge_pipeline.fit(X_train, y_train)
predictions = ridge_pipeline.predict(X_test)
print(f"R² Score: {r2_score(y_test, predictions):.4f}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, predictions):.2f}")
print(f"Mean Squared Error: {mean_squared_error(y_test, predictions):.2f}")


R² Score: 0.9033
Mean Absolute Error: 1.43
Mean Squared Error: 5.54


lasso

In [ ]:
# training lasso
lasso_pipeline = Pipeline(
    steps=[
        ('lasso', Lasso(alpha=1.0))
    ]
)
lasso_pipeline.fit(X_train, y_train)
predictions = lasso_pipeline.predict(X_test)
print(f"R² Score: {r2_score(y_test, predictions):.4f}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, predictions):.2f}")
print(f"Mean Squared Error: {mean_squared_error(y_test, predictions):.2f}")

R² Score: 0.3275
Mean Absolute Error: 4.34
Mean Squared Error: 38.53


ElasticNet

In [ ]:
# ElasticNet
elasticnet_pipeline = Pipeline(
    steps=[
        ('elasticnet', ElasticNet(alpha=1.0, l1_ratio=0.5))
    ]
)
elasticnet_pipeline.fit(X_train, y_train)
predictions = elasticnet_pipeline.predict(X_test)
print(f"R² Score: {r2_score(y_test, predictions):.4f}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, predictions):.2f}")
print(f"Mean Squared Error: {mean_squared_error(y_test, predictions):.2f}")

R² Score: 0.2110
Mean Absolute Error: 4.74
Mean Squared Error: 45.21


LinerReg

In [ ]:
# linear regression
lr = Pipeline(
    steps=[
        ('linear', LinearRegression())
    ]
)
lr.fit(X_train, y_train)
predictions = lr.predict(X_test)
print(f"R² Score: {r2_score(y_test, predictions):.4f}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, predictions):.2f}")
print(f"Mean Squared Error: {mean_squared_error(y_test, predictions):.2f}")

R² Score: 0.8460
Mean Absolute Error: 1.67
Mean Squared Error: 8.82


RandomForestRegressor

In [ ]:
# Random Forest Regressor
rf = Pipeline(
    steps=[
        ('random_forest', RandomForestRegressor(n_estimators=100, random_state=42))
    ]
)
rf.fit(X_train, y_train)
predictions = rf.predict(X_test)
print(f"R² Score: {r2_score(y_test, predictions):.4f}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, predictions):.2f}")
print(f"Mean Squared Error: {mean_squared_error(y_test, predictions):.2f}")

R² Score: 0.9217
Mean Absolute Error: 1.14
Mean Squared Error: 4.49


GradientBoostingRegressor

In [ ]:
# GradientBoostingRegressor
gbr = Pipeline(
    steps=[

        ('gradientboosting', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3))
    ]
)
gbr.fit(X_train, y_train)
predictions = gbr.predict(X_test)
print(f"R² Score: {r2_score(y_test, predictions):.4f}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, predictions):.2f}")
print(f"Mean Squared Error: {mean_squared_error(y_test, predictions):.2f}")

R² Score: 0.9072
Mean Absolute Error: 1.42
Mean Squared Error: 5.32


DecisionTreeRegressor

In [ ]:
# DecisionTreeRegressor
dtr = Pipeline(
    steps=[
        ('decision_tree', DecisionTreeRegressor(random_state=42))
    ]
)
dtr.fit(X_train, y_train)
predictions = dtr.predict(X_test)
print(f"R² Score: {r2_score(y_test, predictions):.4f}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, predictions):.2f}")
print(f"Mean Squared Error: {mean_squared_error(y_test, predictions):.2f}")

R² Score: 0.8602
Mean Absolute Error: 1.48
Mean Squared Error: 8.01


SVR

In [ ]:
# SVR
svr = Pipeline(
    steps=[
        ('svr', SVR(kernel='rbf', C=1.0, epsilon=0.1))
    ]
)
svr.fit(X_train, y_train)
predictions = svr.predict(X_test)
print(f"R² Score: {r2_score(y_test, predictions):.4f}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, predictions):.2f}")
print(f"Mean Squared Error: {mean_squared_error(y_test, predictions):.2f}")

R² Score: 0.8048
Mean Absolute Error: 1.72
Mean Squared Error: 11.18


XGBRegressor

In [ ]:
# XGBRegressor
xgb = Pipeline(
    steps=[
        ('xgb', XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42))
    ]
)
xgb.fit(X_train, y_train)
predictions = xgb.predict(X_test)
print(f"R² Score: {r2_score(y_test, predictions):.4f}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, predictions):.2f}")
print(f"Mean Squared Error: {mean_squared_error(y_test, predictions):.2f}")

R² Score: 0.9126
Mean Absolute Error: 1.38
Mean Squared Error: 5.01
